# Font Style Transfer — End-to-End Colab Pipeline

Few-shot font style transfer. Given a few reference glyphs in some style, generate any other character in that style.

**Pipeline:**
1. Setup (install + GPU check)
2. Data synthesis (download Google Fonts + render) + synthesis report
3. Train / val split
4. Model architecture (U-Net + Style Transformer + Multi-task D)
5. Dataset class
6. Training (VGG perceptual + EMA + R1 GP + bf16)
7. Visualise training samples
8. Evaluation on held-out splits
9. Inference demo

**Recommended runtime:** `Runtime → Change runtime type → T4 GPU` (free) or `A100` (paid).

Mount Drive in cell 2 if you want checkpoints to survive Colab restarts.


In [ ]:
# === 0. Setup ===
!pip install -q tqdm pillow requests

import os, sys, json, random, math, copy, time
from pathlib import Path
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Optional: mount Drive for persistent checkpoints ---
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT_DIR = "/content/drive/MyDrive/font_generation"

ROOT_DIR = "/content/font_generation"
os.makedirs(ROOT_DIR, exist_ok=True)
os.chdir(ROOT_DIR)
print(f"Working dir: {os.getcwd()}")


## 1. Data Synthesis

Downloads fonts from `google/fonts` GitHub and renders each character to a 128×128 grayscale PNG.

**Configure below:**
- `MAX_FONTS`: 100 (smoke), 500 (recommended), 1000 (best quality)
- `CHARS`: A–Z, a–z, 0–9 = 62 characters

**Strict cleanup policy.** Any font that does not render **every** character in `CHARS` is deleted entirely (.ttf + image folder). The training set therefore contains only fonts with uniform glyph coverage — no partial fonts, no missing characters.

You can relax this by passing `min_chars_per_font=<N>` to the render function (any font rendering ≥ N glyphs is kept). The default (`None`) requires all `len(CHARS)` glyphs.

**GitHub token (optional but recommended)** to skip the anonymous 60 req/h rate limit.
In Colab: click the **key icon** in the sidebar → add a secret `GITHUB_TOKEN` with your PAT.

A JSON report at `data/synthesis_report.json` lists every font that was downloaded, failed PIL load, or got deleted as incomplete. The diagnostic cell right after render prints these as a table.


In [ ]:
# === 1. Data synthesis ===
import csv, itertools, requests
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm

# ---- Config (tweak here) ----
MAX_FONTS = 500
CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
IMAGE_SIZE = 128
FONT_SIZE = 96
MIN_CHARS_PER_FONT = 6   # a font is dropped from the dataset below this

FONT_DIR = Path("data/fonts")
IMAGE_DIR = Path("data/images")
LABELS_CSV = Path("data/labels.csv")
MAPPING_CSV = Path("data/char_mapping.csv")
REPORT_JSON = Path("data/synthesis_report.json")

# ---- GitHub token from Colab Secrets, env var, or empty ----
try:
    from google.colab import userdata
    GITHUB_TOKEN = (userdata.get('GITHUB_TOKEN') or '').strip()
except Exception:
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '').strip()
print(f"Using GitHub token: {'yes' if GITHUB_TOKEN else 'no (rate-limited)'}")

GOOGLE_FONTS_TREE_API = "https://api.github.com/repos/google/fonts/git/trees/main?recursive=1"
GOOGLE_FONTS_RAW_BASE = "https://raw.githubusercontent.com/google/fonts/main/"

def github_headers():
    return {"Authorization": f"Bearer {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

def safe_name(value):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in value).strip("_")

def glyph_exists(font, ch):
    try:
        return font.getmask(ch).getbbox() is not None
    except Exception:
        return False

def list_google_font_paths(session):
    response = session.get(GOOGLE_FONTS_TREE_API, timeout=60)
    response.raise_for_status()
    tree = response.json().get("tree", [])
    out = []
    for item in tree:
        if item.get("type") != "blob":
            continue
        path = item.get("path", "")
        if not (path.endswith(".ttf") or path.endswith(".otf")):
            continue
        if "apache/" in path or "ofl/" in path or "ufl/" in path:
            out.append(path)
    return out

def download_google_fonts(font_dir, max_fonts=0):
    font_dir.mkdir(parents=True, exist_ok=True)
    downloaded, skipped_existing, failed_http = [], [], []
    with requests.Session() as session:
        session.headers.update(github_headers())
        paths = list_google_font_paths(session)
        if max_fonts > 0:
            paths = paths[:max_fonts]
        for path in tqdm(paths, desc="Downloading fonts"):
            out = font_dir / path.replace("/", "_")
            if out.exists():
                skipped_existing.append(out.name)
                continue
            r = session.get(GOOGLE_FONTS_RAW_BASE + path, timeout=60)
            if r.status_code == 200:
                out.write_bytes(r.content)
                downloaded.append(out.name)
            else:
                failed_http.append({"path": path, "status": r.status_code})
    return {
        "downloaded": downloaded,
        "skipped_existing": skipped_existing,
        "failed_http": failed_http,
    }

dl_report = download_google_fonts(FONT_DIR, max_fonts=MAX_FONTS)
print(f"Downloaded: {len(dl_report['downloaded'])}  "
      f"already-cached: {len(dl_report['skipped_existing'])}  "
      f"HTTP errors: {len(dl_report['failed_http'])}")
if dl_report['failed_http']:
    print("First 5 HTTP failures:")
    for x in dl_report['failed_http'][:5]:
        print(f"  {x['status']}  {x['path']}")


In [ ]:
import shutil

def render_character_images(font_dir, image_dir, labels_csv, mapping_csv,
                              chars, image_size=128, font_size=96,
                              cleanup=True, min_chars_per_font=None):
    """Render every (font, char) pair. With cleanup=True:
       A) fonts PIL cannot load          → .ttf deleted
       B+C) fonts rendering fewer than    → .ttf + image folder deleted
            ``min_chars_per_font`` glyphs (defaults to len(chars) — STRICT)
       Only fonts that render every requested char remain in labels.csv.
    """
    if min_chars_per_font is None:
        min_chars_per_font = len(chars)

    image_dir.mkdir(parents=True, exist_ok=True)
    rows, index_by_char = [], {}
    font_files = sorted(list(font_dir.glob("*.ttf")) + list(font_dir.glob("*.otf")))

    failed_pil_load = []          # case A: PIL ImageFont.truetype raised
    deleted_ttf_pil_fail = []     # case A: .ttf removed
    deleted_incomplete = []       # case B+C: .ttf + folder removed
    glyphs_per_font = {}          # font_name -> count (only complete fonts kept)
    font_to_path = {}

    for font_path in tqdm(font_files, desc="Rendering"):
        font_name = safe_name(font_path.stem)
        font_to_path[font_name] = font_path
        try:
            font = ImageFont.truetype(str(font_path), font_size)
        except (OSError, IOError) as e:
            failed_pil_load.append({"file": font_path.name, "error": str(e)[:80]})
            if cleanup:
                try:
                    font_path.unlink()
                    deleted_ttf_pil_fail.append(font_path.name)
                except OSError:
                    pass
            continue
        out_dir = image_dir / font_name
        out_dir.mkdir(parents=True, exist_ok=True)
        per_font_count = 0
        for ch in chars:
            if not glyph_exists(font, ch):
                continue
            img = Image.new("L", (image_size, image_size), color=255)
            draw = ImageDraw.Draw(img)
            bbox = draw.textbbox((0, 0), ch, font=font)
            if bbox is None:
                continue
            w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
            x = (image_size - w) // 2 - bbox[0]
            y = (image_size - h) // 2 - bbox[1]
            draw.text((x, y), ch, font=font, fill=0)
            code_str = f"U+{ord(ch):04X}"
            image_path = out_dir / f"{code_str}.png"
            img.save(image_path)
            row = {"label_character": ch, "font_name": font_name, "picture_path": image_path.as_posix()}
            rows.append(row)
            index_by_char.setdefault(ch, []).append(row)
            per_font_count += 1
        glyphs_per_font[font_name] = per_font_count

    # === Strict cleanup: delete any font missing chars ===
    if cleanup:
        bad = set()
        for font_name, count in list(glyphs_per_font.items()):
            if count < min_chars_per_font:
                out_dir = image_dir / font_name
                if out_dir.exists():
                    shutil.rmtree(out_dir, ignore_errors=True)
                ttf = font_to_path.get(font_name)
                if ttf and ttf.exists():
                    try:
                        ttf.unlink()
                    except OSError:
                        pass
                deleted_incomplete.append({
                    "font": font_name,
                    "rendered": count,
                    "required": min_chars_per_font,
                })
                bad.add(font_name)
        if bad:
            rows = [r for r in rows if r["font_name"] not in bad]
            for ch in list(index_by_char.keys()):
                index_by_char[ch] = [r for r in index_by_char[ch] if r["font_name"] not in bad]
                if not index_by_char[ch]:
                    del index_by_char[ch]
            for name in bad:
                glyphs_per_font.pop(name, None)

    labels_csv.parent.mkdir(parents=True, exist_ok=True)
    with labels_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["label_character", "font_name", "picture_path"])
        w.writeheader(); w.writerows(rows)

    mapping_rows = []
    for ch, items in index_by_char.items():
        for src, tgt in itertools.combinations(items, 2):
            mapping_rows.append({
                "label_character": ch,
                "source_font": src["font_name"], "source_picture_path": src["picture_path"],
                "target_font": tgt["font_name"], "target_picture_path": tgt["picture_path"],
            })
    with mapping_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["label_character", "source_font", "source_picture_path", "target_font", "target_picture_path"])
        w.writeheader(); w.writerows(mapping_rows)

    return {
        "font_files_scanned": len(font_files),
        "complete_fonts": len(glyphs_per_font),
        "labels_rows": len(rows),
        "mapping_rows": len(mapping_rows),
        "required_glyphs": min_chars_per_font,
        "failed_pil_load": failed_pil_load,
        "deleted_ttf_pil_fail": deleted_ttf_pil_fail,
        "deleted_incomplete": deleted_incomplete,
        "glyphs_per_font": glyphs_per_font,
    }

render_stats = render_character_images(
    FONT_DIR, IMAGE_DIR, LABELS_CSV, MAPPING_CSV, CHARS, IMAGE_SIZE, FONT_SIZE,
    cleanup=True,                       # delete fonts that fail or are incomplete
    min_chars_per_font=None,            # None = require all len(CHARS) glyphs (STRICT)
)

REPORT_JSON.parent.mkdir(parents=True, exist_ok=True)
REPORT_JSON.write_text(json.dumps({"download": dl_report, "render": render_stats}, indent=2, ensure_ascii=False))
print(f"Wrote {REPORT_JSON}")
print(f"Scanned: {render_stats['font_files_scanned']}  "
      f"complete kept: {render_stats['complete_fonts']}  "
      f"(required {render_stats['required_glyphs']} glyphs each)")
print(f"Deleted PIL-fail: {len(render_stats['deleted_ttf_pil_fail'])}  "
      f"Deleted incomplete: {len(render_stats['deleted_incomplete'])}")


### 1b. Synthesis diagnostic

Inspect exactly **which fonts were dropped** and why. The report is loaded from `data/synthesis_report.json` so you can re-run this cell any time without redoing synthesis.


In [ ]:
# === 1b. Synthesis diagnostic ===
report = json.loads(REPORT_JSON.read_text())
dl = report["download"]
rd = report["render"]

print("=" * 70)
print("DOWNLOAD")
print("=" * 70)
print(f"  Downloaded     : {len(dl['downloaded'])}")
print(f"  Already cached : {len(dl['skipped_existing'])}")
print(f"  HTTP failures  : {len(dl['failed_http'])}")
if dl['failed_http']:
    print("\n  HTTP failures (up to 10):")
    for x in dl['failed_http'][:10]:
        print(f"    {x['status']:>4}  {x['path']}")

print()
print("=" * 70)
print("RENDER")
print("=" * 70)
print(f"  Font files scanned   : {rd['font_files_scanned']}")
print(f"  Complete fonts kept  : {rd['complete_fonts']}  (required {rd['required_glyphs']} glyphs)")
print(f"  Total labels rows    : {rd['labels_rows']}")

print()
print("=" * 70)
print("CLEANUP (auto-removed)")
print("=" * 70)
print(f"  A) PIL load fail (.ttf deleted)         : {len(rd['deleted_ttf_pil_fail'])}")
print(f"  B+C) Incomplete (.ttf + folder deleted) : {len(rd['deleted_incomplete'])}")

if rd['deleted_ttf_pil_fail']:
    print("\n  Deleted .ttf (PIL could not load):")
    for name in rd['deleted_ttf_pil_fail']:
        print(f"    - {name}")

if rd['deleted_incomplete']:
    print(f"\n  Deleted incomplete fonts (rendered count / required {rd['required_glyphs']}):")
    for x in sorted(rd['deleted_incomplete'], key=lambda d: d['rendered']):
        print(f"    {x['rendered']:>3}/{x['required']:<3}  {x['font']}")

# Histogram across kept (complete) fonts only
glyph_counts = list(rd['glyphs_per_font'].values())
if glyph_counts:
    from collections import Counter
    c = Counter(glyph_counts)
    print(f"\n  Glyph-count distribution (kept fonts):")
    for k in sorted(c.keys()):
        bar = "#" * min(c[k], 40)
        print(f"    {k:>3} glyphs:  {c[k]:>4}  {bar}")


## 2. Train / Val Split

Carves out three validation sets to measure generalization to:
- **unseen fonts** (style generalization)
- **unseen characters** (content generalization)
- **both** (hardest test)

Train fonts are what the model sees; val fonts test if the encoder learned _style features_ rather than memorizing specific fonts.


In [ ]:
# === 2. Split dataset ===
from collections import defaultdict

VAL_FONTS = 50
VAL_CHARS = "KQXjz"
SPLIT_SEED = 42

LABELS_TRAIN_CSV = Path("data/labels_train.csv")
LABELS_VAL_FONT_CSV = Path("data/labels_val_unseen_font.csv")
LABELS_VAL_CHAR_CSV = Path("data/labels_val_unseen_char.csv")
LABELS_VAL_BOTH_CSV = Path("data/labels_val_unseen_both.csv")
SPLIT_META = Path("data/split_meta.json")

def split_labels_csv(labels_csv, val_fonts_count, val_chars_str, seed):
    rows = list(csv.DictReader(labels_csv.open(newline="", encoding="utf-8")))
    fonts = sorted({r["font_name"] for r in rows})
    chars = sorted({r["label_character"] for r in rows})
    rng = random.Random(seed)
    val_fonts = set(rng.sample(fonts, min(val_fonts_count, len(fonts) - 1)))
    val_chars = set(val_chars_str) & set(chars)

    buckets = defaultdict(list)
    for r in rows:
        fh = r["font_name"] in val_fonts
        ch = r["label_character"] in val_chars
        if not fh and not ch:
            buckets["train"].append(r)
        elif fh and not ch:
            buckets["val_font"].append(r)
        elif not fh and ch:
            buckets["val_char"].append(r)
        else:
            buckets["val_both"].append(r)

    def write(path, rows_):
        with path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["label_character", "font_name", "picture_path"])
            w.writeheader(); w.writerows(rows_)
    write(LABELS_TRAIN_CSV, buckets["train"])
    write(LABELS_VAL_FONT_CSV, buckets["val_font"])
    write(LABELS_VAL_CHAR_CSV, buckets["val_char"])
    write(LABELS_VAL_BOTH_CSV, buckets["val_both"])

    meta = {
        "seed": seed,
        "total_rows": len(rows), "total_fonts": len(fonts), "total_chars": len(chars),
        "val_fonts": sorted(val_fonts), "val_chars": sorted(val_chars),
        "counts": {k: len(v) for k, v in buckets.items()},
    }
    SPLIT_META.write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    return meta

meta = split_labels_csv(LABELS_CSV, VAL_FONTS, VAL_CHARS, SPLIT_SEED)
print(f"Total: {meta['total_fonts']} fonts × {meta['total_chars']} chars = {meta['total_rows']} rows")
print(f"Held-out: {len(meta['val_fonts'])} fonts | chars={meta['val_chars']}")
for k, c in meta['counts'].items():
    print(f"  {k:<10} {c}")


## 3. Model Architecture

**U-Net Generator + Style Transformer + Multi-task Discriminator**

- *Content path*: U-Net encoder. Skip connections at every level guarantee spatial detail of the input glyph carries to the output (prevents the "blob collapse" the simpler bottleneck-only design suffers from).
- *Style path*: per-reference CNN encoder + self-attention over K refs + CLS token. Produces a single style vector regardless of K.
- *Decoder*: U-Net decoder. At each scale: AdaIN with style code → upsample → concat with content skip → conv.
- *Discriminator*: spectral-norm PatchGAN trunk + two aux heads (font classifier, char classifier). The aux heads force D to learn disentangled features, which is a much stronger signal than real/fake alone.


In [ ]:
# === 3. Model architecture ===
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm="instance", act="lrelu"):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, k, s, p)]
        if norm == "instance": layers.append(nn.InstanceNorm2d(out_ch, affine=True))
        elif norm == "batch": layers.append(nn.BatchNorm2d(out_ch))
        if act == "lrelu": layers.append(nn.LeakyReLU(0.2, inplace=True))
        elif act == "relu": layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, 1, 1); self.n1 = nn.InstanceNorm2d(c, affine=True)
        self.c2 = nn.Conv2d(c, c, 3, 1, 1); self.n2 = nn.InstanceNorm2d(c, affine=True)
    def forward(self, x):
        h = F.leaky_relu(self.n1(self.c1(x)), 0.2)
        h = self.n2(self.c2(h))
        return F.leaky_relu(x + h, 0.2)

class AdaIN(nn.Module):
    def __init__(self, c, style_dim):
        super().__init__()
        self.norm = nn.InstanceNorm2d(c, affine=False)
        self.fc = nn.Linear(style_dim, c * 2)
        nn.init.zeros_(self.fc.bias)
        self.fc.bias.data[:c] = 1.0
        nn.init.normal_(self.fc.weight, std=0.01)
    def forward(self, x, style):
        h = self.norm(x)
        gamma, beta = self.fc(style).chunk(2, dim=1)
        return gamma.unsqueeze(-1).unsqueeze(-1) * h + beta.unsqueeze(-1).unsqueeze(-1)

class AdaINResBlock(nn.Module):
    def __init__(self, c, style_dim):
        super().__init__()
        self.a1 = AdaIN(c, style_dim); self.c1 = nn.Conv2d(c, c, 3, 1, 1)
        self.a2 = AdaIN(c, style_dim); self.c2 = nn.Conv2d(c, c, 3, 1, 1)
    def forward(self, x, style):
        h = self.c1(F.leaky_relu(self.a1(x, style), 0.2))
        h = self.c2(F.leaky_relu(self.a2(h, style), 0.2))
        return x + h

class ContentEncoder(nn.Module):
    def __init__(self, in_ch=1, base=64, depth=4):
        super().__init__()
        self.stem = ConvBlock(in_ch, base, 7, 1, 3)
        self.downs = nn.ModuleList()
        ch = base
        self.channels = [base]
        for _ in range(depth):
            nc = min(ch * 2, 512)
            self.downs.append(nn.Sequential(ConvBlock(ch, nc, 4, 2, 1), ResBlock(nc)))
            ch = nc; self.channels.append(ch)
    def forward(self, x):
        feats = [self.stem(x)]
        for d in self.downs: feats.append(d(feats[-1]))
        return feats

class StyleEncoder(nn.Module):
    def __init__(self, in_ch=1, base=64, n_down=5, max_ch=256, style_dim=256, heads=4):
        super().__init__()
        layers = [ConvBlock(in_ch, base, 7, 1, 3, norm="instance")]
        ch = base
        for _ in range(n_down):
            nc = min(ch * 2, max_ch)
            layers.append(ConvBlock(ch, nc, 4, 2, 1, norm="instance"))
            ch = nc
        self.conv = nn.Sequential(*layers)
        self.proj = nn.Linear(ch, style_dim)
        self.attn = nn.MultiheadAttention(style_dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(style_dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, style_dim))
        nn.init.normal_(self.cls, std=0.02)
        self.style_dim = style_dim
    def _enc_one(self, x):
        h = self.conv(x)
        h = F.adaptive_avg_pool2d(h, 1).flatten(1)
        return self.proj(h)
    def forward(self, refs):
        if refs.dim() == 4: return self._enc_one(refs)
        B, K, C, H, W = refs.shape
        tokens = self._enc_one(refs.view(B*K, C, H, W)).view(B, K, -1)
        cls = self.cls.expand(B, -1, -1)
        seq = torch.cat([cls, tokens], dim=1)
        out, _ = self.attn(seq, seq, seq, need_weights=False)
        out = self.norm(seq + out)
        return out[:, 0]

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, style_dim):
        super().__init__()
        self.a_in = AdaIN(in_ch, style_dim)
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.c1 = nn.Conv2d(in_ch + skip_ch, out_ch, 3, 1, 1)
        self.a_mid = AdaIN(out_ch, style_dim)
        self.c2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.proj = nn.Conv2d(in_ch + skip_ch, out_ch, 1) if (in_ch + skip_ch) != out_ch else nn.Identity()
    def forward(self, x, skip, style):
        h = F.leaky_relu(self.a_in(x, style), 0.2)
        h = self.up(h)
        h = torch.cat([h, skip], dim=1)
        res = self.proj(h)
        h = self.c1(h)
        h = F.leaky_relu(self.a_mid(h, style), 0.2)
        h = self.c2(h)
        return h + res

class Decoder(nn.Module):
    def __init__(self, enc_ch, out_ch=1, style_dim=256, n_bot=3):
        super().__init__()
        rev = list(reversed(enc_ch))
        bn = rev[0]
        self.bot = nn.ModuleList([AdaINResBlock(bn, style_dim) for _ in range(n_bot)])
        self.ups = nn.ModuleList()
        cur = bn
        for skip_ch, tgt_ch in zip(rev[1:], rev[1:]):
            self.ups.append(UpBlock(cur, skip_ch, tgt_ch, style_dim)); cur = tgt_ch
        self.fa = AdaIN(cur, style_dim)
        self.fin = nn.Sequential(
            nn.Conv2d(cur, cur, 3, 1, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(cur, out_ch, 7, 1, 3), nn.Tanh()
        )
    def forward(self, feats, style):
        rev = list(reversed(feats)); h = rev[0]
        for b in self.bot: h = b(h, style)
        for up, sk in zip(self.ups, rev[1:]): h = up(h, sk, style)
        h = F.leaky_relu(self.fa(h, style), 0.2)
        return self.fin(h)

class Generator(nn.Module):
    def __init__(self, image_channels=1, style_dim=256, base=64, depth=4):
        super().__init__()
        self.ce = ContentEncoder(image_channels, base, depth)
        self.se = StyleEncoder(image_channels, base, 5, 256, style_dim)
        self.dec = Decoder(self.ce.channels, image_channels, style_dim)
        self.style_dim = style_dim
    def encode_content(self, x): return self.ce(x)
    def encode_style(self, r): return self.se(r)
    def decode(self, f, s): return self.dec(f, s)
    def forward(self, content, refs): return self.dec(self.ce(content), self.se(refs))

class Discriminator(nn.Module):
    def __init__(self, in_ch=1, n_fonts=0, n_chars=0, base=64, n_down=4, max_ch=512):
        super().__init__()
        self.n_fonts, self.n_chars = n_fonts, n_chars
        trunk = [spectral_norm(nn.Conv2d(in_ch, base, 4, 2, 1)), nn.LeakyReLU(0.2, inplace=True)]
        ch = base
        for _ in range(n_down - 1):
            nc = min(ch * 2, max_ch)
            trunk += [spectral_norm(nn.Conv2d(ch, nc, 4, 2, 1)), nn.LeakyReLU(0.2, inplace=True)]
            ch = nc
        self.trunk = nn.Sequential(*trunk)
        self.patch = spectral_norm(nn.Conv2d(ch, 1, 4, 1, 1))
        self.font_head = nn.Linear(ch, n_fonts) if n_fonts > 0 else None
        self.char_head = nn.Linear(ch, n_chars) if n_chars > 0 else None
    def forward(self, x):
        h = self.trunk(x)
        patch = self.patch(h)
        pooled = F.adaptive_avg_pool2d(h, 1).flatten(1)
        f = self.font_head(pooled) if self.font_head is not None else None
        c = self.char_head(pooled) if self.char_head is not None else None
        return patch, f, c

class VGGPerceptual(nn.Module):
    def __init__(self, layer_idx=(3, 8, 15, 22)):
        super().__init__()
        from torchvision import models
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features.eval()
        for p in vgg.parameters(): p.requires_grad = False
        self.vgg = vgg; self.layer_idx = layer_idx
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def _prep(self, x):
        x = (x + 1.0) / 2.0
        if x.size(1) == 1: x = x.repeat(1, 3, 1, 1)
        return (x - self.mean) / self.std
    def forward(self, x, y):
        x, y = self._prep(x), self._prep(y)
        loss = x.new_tensor(0.0); mx = max(self.layer_idx)
        for i, layer in enumerate(self.vgg):
            x = layer(x); y = layer(y)
            if i in self.layer_idx: loss = loss + F.l1_loss(x, y)
            if i >= mx: break
        return loss / len(self.layer_idx)

_g = Generator().to(DEVICE)
_d = Discriminator(n_fonts=100, n_chars=62).to(DEVICE)
print(f"Generator params: {sum(p.numel() for p in _g.parameters()) / 1e6:.2f}M")
print(f"Discriminator params: {sum(p.numel() for p in _d.parameters()) / 1e6:.2f}M")
del _g, _d


## 4. Dataset Class

For each sample the dataset returns:
- `content_image` — target character rendered in some other (content) font, with light affine augmentation
- `style_images` — K reference glyphs from the target font
- `target_image` — target character rendered in the target font (ground truth)
- `target_font_id`, `char_id` — integer labels for D's aux classifiers


In [ ]:
# === 4. Dataset class ===
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

def load_index(labels_csv):
    font_to_char = defaultdict(dict)
    char_to_fonts = defaultdict(set)
    with Path(labels_csv).open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            font_to_char[r["font_name"]][r["label_character"]] = r["picture_path"]
            char_to_fonts[r["label_character"]].add(r["font_name"])
    return font_to_char, char_to_fonts

class FontPairDataset(Dataset):
    def __init__(self, labels_csv, image_size=128, k_style=4, min_chars_per_font=5, augment=True, seed=None):
        f2c, c2f = load_index(labels_csv)
        self.f2c = {f: chars for f, chars in f2c.items() if len(chars) >= min_chars_per_font + 1}
        self.c2f = {c: {f for f in fs if f in self.f2c} for c, fs in c2f.items()}
        self.fonts = sorted(self.f2c.keys())
        self.chars = sorted({c for chars in self.f2c.values() for c in chars})
        self.font_to_id = {f: i for i, f in enumerate(self.fonts)}
        self.char_to_id = {c: i for i, c in enumerate(self.chars)}
        self.samples = [(f, c) for f, cs in self.f2c.items() for c in cs if len(self.c2f.get(c, ())) >= 2]
        self.k_style = k_style
        self.augment = augment
        self.base_tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        self.aug_tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomAffine(degrees=3, translate=(0.05, 0.05), scale=(0.93, 1.07), fill=255),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        self.rng = random.Random(seed)

    @property
    def n_fonts(self): return len(self.fonts)
    @property
    def n_chars(self): return len(self.chars)
    def __len__(self): return len(self.samples)
    def _load(self, path, aug=False):
        img = Image.open(path).convert("L")
        return (self.aug_tf if (aug and self.augment) else self.base_tf)(img)

    def __getitem__(self, idx):
        tgt_font, c = self.samples[idx]
        cf_candidates = [f for f in self.c2f[c] if f != tgt_font]
        content_font = self.rng.choice(cf_candidates) if cf_candidates else tgt_font
        tgt_chars = list(self.f2c[tgt_font].keys())
        pool = [x for x in tgt_chars if x != c]
        style_chars = (self.rng.sample(pool, self.k_style) if len(pool) >= self.k_style
                       else self.rng.choices(pool or tgt_chars, k=self.k_style))
        return {
            "content_image": self._load(self.f2c[content_font][c], aug=True),
            "style_images": torch.stack([self._load(self.f2c[tgt_font][s]) for s in style_chars], dim=0),
            "target_image": self._load(self.f2c[tgt_font][c]),
            "target_font_id": torch.tensor(self.font_to_id[tgt_font], dtype=torch.long),
            "char_id": torch.tensor(self.char_to_id[c], dtype=torch.long),
        }

train_ds = FontPairDataset(LABELS_TRAIN_CSV, image_size=IMAGE_SIZE, k_style=4)
print(f"Train dataset: {len(train_ds)} samples, {train_ds.n_fonts} fonts, {train_ds.n_chars} chars")


## 5. Training

Loss design that prevents the "blob collapse" of pixel-L1-dominated GANs:

| Loss | Weight | Purpose |
| --- | --- | --- |
| Adversarial (hinge) | 1.0 | D pushes G to be realistic |
| VGG perceptual | 5.0 | Feature-level match (the workhorse) |
| L1 reconstruction | 0.5 | Just anchors pixel intensity |
| Content consistency | 2.0 | E_c(fake) ≈ E_c(content) |
| Style consistency | 1.0 | E_s(fake) ≈ E_s(refs) |
| Font/char classification | 1.0 each | Aux signal for D |
| R1 gradient penalty | 10 (every 16 steps) | D stability |

Training tricks: **EMA generator** (decay 0.999), **TTUR** (D lr > G lr), **bf16 autocast**, Adam betas (0, 0.99).

On a free T4 with batch=16, expect ~3 it/s → ~3 min/epoch on a 500-font dataset → ~3 hours for 60 epochs.


In [ ]:
# === 5. Training setup ===
OUT_DIR = Path("runs/v2_unet"); OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "ckpt").mkdir(exist_ok=True); (OUT_DIR / "samples").mkdir(exist_ok=True)

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
NUM_WORKERS = 2 if DEVICE == "cuda" else 0
EPOCHS = 60
K_STYLE = 4
STYLE_DIM = 256
G_LR, D_LR = 1e-4, 4e-4
BETA1, BETA2 = 0.0, 0.99
EMA_DECAY = 0.999
EMA_WARMUP = 1000
R1_INTERVAL = 16
LAMBDA = dict(adv=1.0, perceptual=5.0, rec=0.5, content=2.0, style=1.0,
              font_cls=1.0, char_cls=1.0, r1=10.0)
USE_BF16 = DEVICE in ("cuda", "mps")

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        if step < EMA_WARMUP:
            for pe, p in zip(self.ema.parameters(), model.parameters()): pe.data.copy_(p.data)
            for be, b in zip(self.ema.buffers(), model.buffers()): be.data.copy_(b.data)
            return
        d = self.decay
        for pe, p in zip(self.ema.parameters(), model.parameters()):
            pe.data.mul_(d).add_(p.data, alpha=1 - d)
        for be, b in zip(self.ema.buffers(), model.buffers()): be.data.copy_(b.data)

def hinge_d(real, fake): return F.relu(1 - real).mean() + F.relu(1 + fake).mean()
def hinge_g(fake): return -fake.mean()
def r1_penalty(d_real, real):
    g = torch.autograd.grad(d_real.sum(), real, create_graph=True, retain_graph=True, only_inputs=True)[0]
    return 0.5 * g.reshape(g.size(0), -1).pow(2).sum(dim=1).mean()

G = Generator(image_channels=1, style_dim=STYLE_DIM).to(DEVICE)
D = Discriminator(n_fonts=train_ds.n_fonts, n_chars=train_ds.n_chars).to(DEVICE)
vgg = VGGPerceptual().to(DEVICE).eval()
opt_g = torch.optim.Adam(G.parameters(), lr=G_LR, betas=(BETA1, BETA2))
opt_d = torch.optim.Adam(D.parameters(), lr=D_LR, betas=(BETA1, BETA2))
ema = EMA(G, decay=EMA_DECAY)

loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                    drop_last=True, pin_memory=(DEVICE == "cuda"))
autocast_kwargs = dict(device_type=DEVICE if DEVICE != "mps" else "cpu",
                       dtype=torch.bfloat16, enabled=USE_BF16)
print(f"Training {EPOCHS} epochs, batch={BATCH_SIZE}, {len(loader)} batches/epoch")


In [ ]:
from torchvision.utils import save_image
from IPython.display import display, clear_output, Image as IPImage

global_step = 0
SAVE_EVERY = 2

for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        content = batch["content_image"].to(DEVICE, non_blocking=True)
        refs = batch["style_images"].to(DEVICE, non_blocking=True)
        target = batch["target_image"].to(DEVICE, non_blocking=True)
        tf_id = batch["target_font_id"].to(DEVICE, non_blocking=True)
        ch_id = batch["char_id"].to(DEVICE, non_blocking=True)

        # D step
        do_r1 = (global_step % R1_INTERVAL == 0)
        real_for_d = target.detach().requires_grad_(do_r1)
        with torch.autocast(**autocast_kwargs):
            with torch.no_grad():
                fake = G(content, refs)
            d_real_p, d_real_f, d_real_c = D(real_for_d)
            d_fake_p, _, _ = D(fake.detach())
            d_adv = hinge_d(d_real_p, d_fake_p)
            d_fc = F.cross_entropy(d_real_f, tf_id)
            d_cc = F.cross_entropy(d_real_c, ch_id)
            d_loss = LAMBDA['adv']*d_adv + LAMBDA['font_cls']*d_fc + LAMBDA['char_cls']*d_cc
        if do_r1:
            d_real_p_fp = D(real_for_d)[0]
            r1 = r1_penalty(d_real_p_fp, real_for_d)
            d_total = d_loss + (LAMBDA['r1'] * R1_INTERVAL) * r1
        else:
            d_total = d_loss
        opt_d.zero_grad(set_to_none=True)
        d_total.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        opt_d.step()

        # G step
        with torch.autocast(**autocast_kwargs):
            feats = G.encode_content(content)
            style = G.encode_style(refs)
            fake = G.decode(feats, style)
            g_fp, g_ff, g_fc = D(fake)
            g_adv = hinge_g(g_fp)
            g_font = F.cross_entropy(g_ff, tf_id)
            g_char = F.cross_entropy(g_fc, ch_id)
            g_vgg = vgg(fake.float(), target.float())
            g_rec = F.l1_loss(fake, target)
            cf = G.encode_content(fake)
            g_con = sum(F.l1_loss(a, b.detach()) for a, b in zip(cf, feats)) / len(feats)
            sf = G.encode_style(fake.unsqueeze(1))
            g_sty = F.l1_loss(sf, style.detach())
            g_loss = (LAMBDA['adv']*g_adv + LAMBDA['perceptual']*g_vgg + LAMBDA['rec']*g_rec
                      + LAMBDA['content']*g_con + LAMBDA['style']*g_sty
                      + LAMBDA['font_cls']*g_font + LAMBDA['char_cls']*g_char)
        opt_g.zero_grad(set_to_none=True)
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_g.step()
        ema.update(G, global_step); global_step += 1

        pbar.set_postfix(adv=f"{g_adv.item():.2f}", vgg=f"{g_vgg.item():.2f}",
                         rec=f"{g_rec.item():.3f}", con=f"{g_con.item():.3f}",
                         d=f"{d_adv.item():.2f}")

    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
        ema.ema.eval()
        with torch.no_grad():
            vis_n = min(8, content.size(0))
            with torch.autocast(**autocast_kwargs):
                fake_vis = ema.ema(content[:vis_n], refs[:vis_n]).float()
        grid = torch.cat([content[:vis_n].cpu(), refs[:vis_n, 0].cpu(),
                          fake_vis.cpu(), target[:vis_n].cpu()], dim=0)
        sample_path = OUT_DIR / "samples" / f"epoch_{epoch+1:03d}.png"
        save_image((grid + 1) / 2, sample_path, nrow=vis_n)
        torch.save({
            "epoch": epoch + 1, "global_step": global_step,
            "G": G.state_dict(), "G_ema": ema.ema.state_dict(),
            "D": D.state_dict(), "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(),
            "vocab": {"n_fonts": train_ds.n_fonts, "n_chars": train_ds.n_chars,
                      "fonts": train_ds.fonts, "chars": train_ds.chars},
        }, OUT_DIR / "ckpt" / "latest.pt")
        clear_output(wait=True)
        print(f"Epoch {epoch+1} saved.  Sample:")
        display(IPImage(str(sample_path)))

print("Training complete.")


## 6. Visualize training samples

Open any saved grid. Each grid has 4 rows (top → bottom):

1. **content** — the character we asked for, in some source font
2. **style ref** — one reference glyph from the target font
3. **generated** ★ — what the model produced
4. **ground truth** — the correct answer

A healthy model has row 3 ≈ row 4.


In [ ]:
# === 6. Visualize ===
from IPython.display import Image as IPImage, display
sample_files = sorted((OUT_DIR / "samples").glob("epoch_*.png"))
print(f"{len(sample_files)} sample grids saved.")
if sample_files:
    display(IPImage(str(sample_files[-1])))


## 7. Evaluation on held-out splits

Loads the EMA generator from the latest checkpoint and computes mean L1 + perceptual error on each validation split. Generalization is good if the three val numbers are close to (or only slightly worse than) the training error.


In [ ]:
# === 7. Evaluation ===
@torch.no_grad()
def evaluate_split(G_ema, vgg_loss, labels_csv, n_samples=256, k_style=4):
    ds = FontPairDataset(labels_csv, image_size=IMAGE_SIZE, k_style=k_style, augment=False, seed=0)
    if len(ds) == 0: return None
    indices = list(range(min(n_samples, len(ds))))
    loader_eval = DataLoader(Subset(ds, indices), batch_size=BATCH_SIZE, shuffle=False)
    l1s, vggs = [], []
    G_ema.eval()
    for batch in loader_eval:
        content = batch["content_image"].to(DEVICE)
        refs = batch["style_images"].to(DEVICE)
        target = batch["target_image"].to(DEVICE)
        fake = G_ema(content, refs)
        l1s.append(F.l1_loss(fake, target).item())
        vggs.append(vgg_loss(fake.float(), target.float()).item())
    return {"l1": sum(l1s)/len(l1s), "vgg": sum(vggs)/len(vggs), "n": len(indices)}

state = torch.load(OUT_DIR / "ckpt" / "latest.pt", map_location=DEVICE)
G_ema = Generator(image_channels=1, style_dim=STYLE_DIM).to(DEVICE)
G_ema.load_state_dict(state["G_ema"])

for name, csv_path in [
    ("train", LABELS_TRAIN_CSV),
    ("val_unseen_font", LABELS_VAL_FONT_CSV),
    ("val_unseen_char", LABELS_VAL_CHAR_CSV),
    ("val_unseen_both", LABELS_VAL_BOTH_CSV),
]:
    if not csv_path.exists():
        continue
    m = evaluate_split(G_ema, vgg, csv_path, n_samples=256)
    if m: print(f"{name:<22} n={m['n']:>4}  L1={m['l1']:.4f}  VGG={m['vgg']:.4f}")


## 8. Inference Demo

Upload K reference glyph PNGs (single character per file), then ask the model to generate any characters in that style.

In Colab use `files.upload()`, or just pass paths if you already saved test images to Drive.


In [ ]:
# === 8. Inference ===
@torch.no_grad()
def transfer(G_ema, style_paths, content_chars, neutral_font_path):
    tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ])
    style_imgs = torch.stack([tf(Image.open(p).convert("L")) for p in style_paths], dim=0)
    style_imgs = style_imgs.unsqueeze(0).to(DEVICE)
    style_code = G_ema.encode_style(style_imgs)

    neutral_font = ImageFont.truetype(str(neutral_font_path), FONT_SIZE)
    outputs = {}
    for ch in content_chars:
        if ch.isspace(): continue
        img = Image.new("L", (IMAGE_SIZE, IMAGE_SIZE), color=255)
        draw = ImageDraw.Draw(img)
        bbox = draw.textbbox((0, 0), ch, font=neutral_font)
        w_, h_ = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x = (IMAGE_SIZE - w_) // 2 - bbox[0]
        y = (IMAGE_SIZE - h_) // 2 - bbox[1]
        draw.text((x, y), ch, font=neutral_font, fill=0)
        content_t = tf(img).unsqueeze(0).to(DEVICE)
        feats = G_ema.encode_content(content_t)
        fake = G_ema.decode(feats, style_code).squeeze(0)
        out = ((fake.clamp(-1, 1) + 1) / 2 * 255).byte().cpu().numpy()
        outputs[ch] = Image.fromarray(out[0], mode="L")
    return outputs

candidates = sorted(list(FONT_DIR.glob("*.ttf")) + list(FONT_DIR.glob("*.otf")))
preferred = ["NotoSans-Regular", "Roboto-Regular", "OpenSans-Regular", "DejaVuSans"]
NEUTRAL_FONT = None
for kw in preferred:
    for c in candidates:
        if kw.lower() in c.name.lower(): NEUTRAL_FONT = c; break
    if NEUTRAL_FONT: break
if NEUTRAL_FONT is None:
    for c in candidates:
        try:
            ImageFont.truetype(str(c), FONT_SIZE); NEUTRAL_FONT = c; break
        except OSError:
            continue
print(f"Neutral content font: {NEUTRAL_FONT}")

val_meta = json.loads(SPLIT_META.read_text())
ref_font_name = val_meta["val_fonts"][0]
ref_dir = IMAGE_DIR / ref_font_name
ref_paths = sorted([p for p in ref_dir.glob("*.png") if p.stem not in ("U+004B","U+0051","U+0058","U+006A","U+007A")])[:4]
print(f"Style refs from unseen font: {ref_font_name}")
print(f"Ref files: {[p.name for p in ref_paths]}")

TARGET_TEXT = "Hello123"
results = transfer(G_ema, ref_paths, TARGET_TEXT, NEUTRAL_FONT)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(results), figsize=(2 * len(results), 2))
if len(results) == 1: axes = [axes]
for ax, (ch, img) in zip(axes, results.items()):
    ax.imshow(img, cmap="gray"); ax.set_title(ch); ax.axis("off")
plt.tight_layout(); plt.show()


---

## Tips

- **Save checkpoints to Drive**: uncomment the `drive.mount` block in cell 2 and point `ROOT_DIR` at your Drive folder.
- **Inspect synthesis failures any time**: re-run cell 1b after the report JSON exists.
- **Resume training**: load `runs/v2_unet/ckpt/latest.pt` into `G`, `D`, `opt_g`, `opt_d`, `ema.ema` before re-entering the training loop.
- **Bigger dataset = better generalization**: bump `MAX_FONTS` from 500 → 1000+ once you have hours to spare.
- **Smaller image size**: dropping from 128 to 64 makes training 4× faster — useful for prototyping.
